Your task is to create a bert-base-classifier of vacancy areas based on their titles.

Each vacancy can have more than one area so it's **Multi-label classification** not Multiclass classification




In [6]:
!pip install transformers==4.44.2 --no-deps
print("Transformers downgraded to 4.44.2. Please restart the kernel now (Run > Restart Session) and proceed to Step 2.")

Transformers downgraded to 4.44.2. Please restart the kernel now (Run > Restart Session) and proceed to Step 2.


In [13]:
!pip install tokenizers==0.19.1 --force-reinstall
print("Tokenizers installed to compatible version 0.19.1. Please restart the kernel now (Run > Restart Session) and proceed to Step 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.6/151.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [15]:
# Core imports (post-downgrade)
import pandas as pd
import numpy as np
import random
import os
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from string import punctuation
from transformers import BertTokenizer  # Now compatible in 4.44.2

# Reload dataset
df = pd.read_csv('/kaggle/input/dataset-2020-csv/dataset_2020.csv')
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

# Text cleaning and seeding
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~')
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

def seed_everything(seed_value=12):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(12)

# Model parameters
MODEL_NAME = 'bert-base-uncased'
MAX_SEQ_LENGTH = 128

# Tokenizer loading (now error-free in 4.44.2)
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# Binarizer
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer = MultiLabelBinarizer()
binarizer.fit(labels_train)

print(f"Setup complete: Tokenizer loaded ({MODEL_NAME}), {len(binarizer.classes_)} unique labels")

Setup complete: Tokenizer loaded (bert-base-uncased), 29 unique labels


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
import os
print("Available input directories:")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

Available input directories:
input/
  dataset-2020-csv/
    dataset_2020.csv


In [10]:
# Load dataset with confirmed path
df = pd.read_csv('/kaggle/input/dataset-2020-csv/dataset_2020.csv')
print(f"Dataset shape: {df.shape}")
print(f"Sample titles:\n{df.head(3)}")
print(f"Unique areas: {df['area'].nunique()}")
print(f"Sample areas: {df['area'].head(3).tolist()}")

# Split data (80% train, 10% valid, 10% test)
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)
print(f"Train: {df_train.shape}, Valid: {df_valid.shape}, Test: {df_test.shape}")

Dataset shape: (78909, 2)
Sample titles:
                                      title        area
0  Expert Java Developer (Technical Leader)  programmer
1           Software Engineer (JVM Runtime)  programmer
2                             PHP developer  programmer
Unique areas: 49
Sample areas: ['programmer', 'programmer', 'programmer']
Train: (56814, 2), Valid: (14204, 2), Test: (7891, 2)


In [11]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [16]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler, Dataset, SequentialSampler
import random
import transformers
import random
import os
import torch
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import Dataset
from nltk.tokenize import word_tokenize
from transformers import BertTokenizer

# Try two or more different bert-like models(different berts, robertas etc. or any other transformer based model) (**2 points max**)
 your notebook should contain the training process of all your models!

In [32]:
MODEL_NAME = 'bert-base-uncased'
MAX_SEQ_LENGTH = 128
RESULT_MODEL_PATH = './model.pt'

In [33]:
def seed_everything(seed_value=12):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(12)

In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [35]:
device

device(type='cuda')

In [36]:
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~') # убрал #

In [37]:
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

In [ ]:
!pip install -U transformers huggingface_hub
print("Libraries updated. Please restart the kernel now (Runtime > Restart Session) and proceed to Step 2.")

Each vacancy can have more than one area separated be space

Exapmle:

Malware Analyst for Imunify Security,analyst it_security

In [22]:
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

# Finish TextClassificationDataset (**1 point max**)

In [23]:
class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, binarizer):
        self.data = data
        self.tokenizer = tokenizer
        self.sentences = [clean(sent) for sent in data.title.tolist()]
        self.target = [labels.split() for labels in data.area.tolist()]
        self.binarizer = binarizer
        self.target_one_hot = torch.tensor(self.binarizer.transform(self.target), dtype=torch.float)
        self.encodings = self.tokenizer(
            self.sentences,
            truncation=True,
            padding=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors='pt'
        )

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.target_one_hot[idx]
        return item

# Test
test_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
print(f"Dataset length: {len(test_dataset)}")
sample = test_dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"Sample input_ids shape: {sample['input_ids'].shape}")
print(f"Sample labels shape: {sample['labels'].shape}")

Dataset length: 56814
Sample keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Sample input_ids shape: torch.Size([30])
Sample labels shape: torch.Size([29])


In [19]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer = MultiLabelBinarizer()
binarizer.fit(labels_train)

print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Number of unique labels: {len(binarizer.classes_)}")
print(f"Sample binarized label: {binarizer.transform([df_train.area.iloc[0].split()])[0]}")  # First label as binary vector

Tokenizer loaded: bert-base-uncased
Number of unique labels: 29
Sample binarized label: [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [24]:
from torch.utils.data import DataLoader, RandomSampler

batch_size = 16

# Create datasets
train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)

# Create dataloaders
train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size)

valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

print(f"DataLoaders created: Train batches: {len(train_dataloader)}, Valid batches: {len(valid_dataloader)}, Test batches: {len(test_dataloader)}")

DataLoaders created: Train batches: 3551, Valid batches: 888, Test batches: 494


In [38]:
import torch.nn as nn
from transformers import BertModel

class BertForMultilabel(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()
        self.num_labels = num_labels
        self.bert = BertModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)  # Dropout for regularization
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)  # Project to label space (768 -> 49)

    def train_bert(self, train_bert_flag=True):
        for param in self.bert.parameters():
            param.requires_grad = train_bert_flag

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.pooler_output  # [CLS] token representation
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# Instantiate model
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)  # Move to GPU

print(f"Model instantiated: {MODEL_NAME}, {num_labels} labels, parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model device: {next(model.parameters()).device}")

Model instantiated: bert-base-uncased, 29 labels, parameters: 109,504,541
Model device: cuda:0


In [ ]:
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)
;

# Train your classifier with freezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch


In [46]:
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import f1_score
import numpy as np

def train(model, iterator, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in tqdm(iterator, desc="Training"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch.get('token_type_ids', None)
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping for stability
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(iterator)



In [48]:
def validate(model, iterator, criterion):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in tqdm(iterator, desc="Validation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch.get('token_type_ids', None)
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).cpu().int().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(iterator)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    f1 = f1_score(all_labels, all_preds, average='micro')
    return avg_loss, f1, all_preds  # Return predictions for test use

In [49]:
def logits_to_labels(logits):
    preds = nn.Sigmoid()(logits.view(-1, num_labels))
    preds = preds.to('cpu').numpy()>0.5
    return preds.tolist()

In [50]:
model.train_bert(False)

In [ ]:
epochs = # ToDo
criterion = # ToDo what criterion do you need for multilabel classification?
optimizer  = # ToDo use adam optimizer
scheduler = # ToDo use StepLR scheduler

In [51]:
epochs = 3
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)  # Optimize only classifier
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

RESULT_MODEL_PATH = './bert_frozen.pt'

# Training loop (corrected unpacking)
best_val_loss = float('inf')
for epoch in range(epochs):
    train_loss = train(model, train_dataloader, optimizer, criterion)
    val_loss, val_f1, _ = validate(model, valid_dataloader, criterion)  # Ignore preds here
    print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)
    scheduler.step()

print("Frozen BERT training complete. Best model saved to bert_frozen.pt.")

Validation: 100%|██████████| 888/888 [00:25<00:00, 34.86it/s]


Epoch 1/3 - Train Loss: 0.0635, Val Loss: 0.0483, Val F1: 0.7595


Validation: 100%|██████████| 888/888 [00:25<00:00, 34.91it/s]


Epoch 2/3 - Train Loss: 0.0571, Val Loss: 0.0459, Val F1: 0.7428


Validation: 100%|██████████| 888/888 [00:25<00:00, 34.95it/s]


Epoch 3/3 - Train Loss: 0.0566, Val Loss: 0.0457, Val F1: 0.7513
Frozen BERT training complete. Best model saved to bert_frozen.pt.


In [52]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device)))
test_preds = validate(model, test_dataloader, criterion)

Validation: 100%|██████████| 494/494 [00:13<00:00, 36.05it/s]


In [57]:
from sklearn.metrics import classification_report  # Add this import
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       0.00      0.00      0.00        61
        analyst       1.00      0.63      0.77       302
    architector       0.00      0.00      0.00       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.00      0.00      0.00       136
 data_scientist       0.00      0.00      0.00       154
       designer       0.99      0.49      0.65       409
devel_metodolog       0.00      0.00      0.00        44
         devops       1.00      0.58      0.73       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       0.00      0.00      0.00        42
        manager       0.91      0.05      0.09       427
       networks       0.00    

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# Train your classifier with unfreezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch

In [62]:
# Unfreeze BERT parameters
model.train_bert(True)

# Hyperparameters for unfrozen phase
epochs = 5
lr = 2e-5
WARMUP_PROPORTION = 0.1
warmup_steps = int(len(train_dataloader) * epochs * WARMUP_PROPORTION)
t_total = len(train_dataloader) * epochs

no_decay = ['bias', 'LayerNorm.weight']  # Parameters exempt from weight decay
param_optimizer = list(model.named_parameters())
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]

criterion = nn.BCEWithLogitsLoss()
optimizer = transformers.AdamW(optimizer_grouped_parameters, lr=lr)
scheduler = transformers.get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total)

# Define save path (ensures variable is in scope)
RESULT_MODEL_PATH_UNFROZEN = './bert_unfrozen.pt'

print(f"Unfrozen setup complete: LR {lr}, Warmup steps {warmup_steps}, Total steps {t_total}")

# Training loop
best_val_loss = float('inf')
for epoch in range(epochs):
    train_loss = train(model, train_dataloader, optimizer, criterion)
    val_loss, val_f1, _ = validate(model, valid_dataloader, criterion)
    print(f'Unfrozen Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH_UNFROZEN)
    scheduler.step()

print("Unfrozen BERT training complete. Best model saved to bert_unfrozen.pt.")

Unfrozen setup complete: LR 2e-05, Warmup steps 1775, Total steps 17755


Validation: 100%|██████████| 888/888 [00:25<00:00, 34.85it/s]


Unfrozen Epoch 1/5 - Train Loss: 0.0565, Val Loss: 0.0458, Val F1: 0.7522


Validation: 100%|██████████| 888/888 [00:25<00:00, 34.89it/s]


Unfrozen Epoch 2/5 - Train Loss: 0.0521, Val Loss: 0.0387, Val F1: 0.8093


...

Unfrozen Epoch 3/5 - Train Loss: 0.0448, Val Loss: 0.0329, Val F1: 0.8446


Validation: 100%|██████████| 888/888 [00:25<00:00, 34.87it/s]


Unfrozen Epoch 4/5 - Train Loss: 0.0385, Val Loss: 0.0278, Val F1: 0.8727


Validation: 100%|██████████| 888/888 [00:25<00:00, 34.87it/s]


Unfrozen Epoch 5/5 - Train Loss: 0.0331, Val Loss: 0.0235, Val F1: 0.8974
Unfrozen BERT training complete. Best model saved to bert_unfrozen.pt.


In [66]:
model.train_bert(True)

In [67]:
from sklearn.metrics import classification_report
import pandas as pd

# Load best unfrozen model
model.load_state_dict(torch.load(RESULT_MODEL_PATH_UNFROZEN, map_location=device))

# Test evaluation
test_loss_unfrozen, test_f1_unfrozen, test_preds_unfrozen = validate(model, test_dataloader, criterion)
print(f'Unfrozen Test Loss: {test_loss_unfrozen:.4f}, Test Micro-F1: {test_f1_unfrozen:.4f}')

# Unfrozen classification report
test_labels_bin = binarizer.transform(test_dataset.target)
print(classification_report(test_labels_bin, test_preds_unfrozen, target_names=binarizer.classes_, zero_division=0))

# Comparison table (assuming frozen metrics from prior run; update with your values if needed)
comparison = pd.DataFrame({
    'Phase': ['Frozen', 'Unfrozen'],
    'Test Loss': [test_loss, test_loss_unfrozen],  # Use variables from frozen eval
    'Micro F1': [test_f1, test_f1_unfrozen]
})
print("\nModel Comparison:")
print(comparison)

Validation: 100%|██████████| 494/494 [00:13<00:00, 35.73it/s]

Unfrozen Test Loss: 0.0233, Test Micro-F1: 0.8966
                 precision    recall  f1-score   support

          admin       1.00      0.15      0.26        61
        analyst       0.98      0.96      0.97       302
    architector       1.00      0.52      0.69       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.84      0.50      0.63       136
 data_scientist       0.98      0.87      0.92       154
       designer       0.97      0.95      0.96       409
devel_metodolog       1.00      0.59      0.74        44
         devops       1.00      0.97      0.98       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       1.00      0.31      0.47        42
        manager       0.76      0.69 

In [68]:
from transformers import RobertaTokenizerFast, RobertaModel

# Update model name for RoBERTa
MODEL_NAME = 'roberta-base'
MAX_SEQ_LENGTH = 128  # Retained for consistency

# Load RoBERTa tokenizer (fast variant for efficiency)
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

# Reload datasets with new tokenizer (binarizer unchanged)
train_dataset_roberta = TextClassificationDataset(df_train, tokenizer, binarizer)
valid_dataset_roberta = TextClassificationDataset(df_valid, tokenizer, binarizer)
test_dataset_roberta = TextClassificationDataset(df_test, tokenizer, binarizer)

# Reload DataLoaders
batch_size = 16
train_sampler = RandomSampler(train_dataset_roberta)
train_dataloader_roberta = DataLoader(train_dataset_roberta, sampler=train_sampler, batch_size=batch_size)
valid_dataloader_roberta = DataLoader(valid_dataset_roberta, batch_size=batch_size)
test_dataloader_roberta = DataLoader(test_dataset_roberta, batch_size=batch_size)

print(f"RoBERTa setup complete: Tokenizer loaded, datasets reloaded with {len(binarizer.classes_)} labels")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

RoBERTa setup complete: Tokenizer loaded, datasets reloaded with 29 labels


In [69]:
class RobertaForMultilabel(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()
        self.num_labels = num_labels
        self.roberta = RobertaModel.from_pretrained(MODEL_NAME)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)  # 768 -> 29

    def train_bert(self, train_bert_flag=True):  # Renamed for generality
        for param in self.roberta.parameters():
            param.requires_grad = train_bert_flag

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        # RoBERTa ignores token_type_ids
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# Instantiate RoBERTa model
num_labels = len(binarizer.classes_)
roberta_model = RobertaForMultilabel(num_labels)
roberta_model.to(device)

print(f"RoBERTa model instantiated: {MODEL_NAME}, {num_labels} labels, parameters: {sum(p.numel() for p in roberta_model.parameters()):,}")
print(f"Model device: {next(roberta_model.parameters()).device}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RoBERTa model instantiated: roberta-base, 29 labels, parameters: 124,667,933
Model device: cuda:0


In [70]:
# Freeze RoBERTa
roberta_model.train_bert(False)

# Hyperparameters (identical to BERT for fair comparison)
epochs = 3
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(roberta_model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.1)

RESULT_MODEL_PATH_ROBERTA_FROZEN = './roberta_frozen.pt'

# Training loop
best_val_loss = float('inf')
for epoch in range(epochs):
    train_loss = train(roberta_model, train_dataloader_roberta, optimizer, criterion)
    val_loss, val_f1, _ = validate(roberta_model, valid_dataloader_roberta, criterion)
    print(f'RoBERTa Frozen Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(roberta_model.state_dict(), RESULT_MODEL_PATH_ROBERTA_FROZEN)
    scheduler.step()

print("RoBERTa frozen training complete.")

Validation: 100%|██████████| 888/888 [00:53<00:00, 16.65it/s]


RoBERTa Frozen Epoch 1/3 - Train Loss: 0.1103, Val Loss: 0.1013, Val F1: 0.0618


Validation: 100%|██████████| 888/888 [00:53<00:00, 16.71it/s]


RoBERTa Frozen Epoch 2/3 - Train Loss: 0.1019, Val Loss: 0.0997, Val F1: 0.5197


Validation: 100%|██████████| 888/888 [00:53<00:00, 16.70it/s]


RoBERTa Frozen Epoch 3/3 - Train Loss: 0.1013, Val Loss: 0.0996, Val F1: 0.5383
RoBERTa frozen training complete.


In [72]:
# Unfreeze RoBERTa
roberta_model.train_bert(True)

# Unfrozen hyperparameters
epochs = 5
lr = 2e-5
WARMUP_PROPORTION = 0.1
warmup_steps = int(len(train_dataloader_roberta) * epochs * WARMUP_PROPORTION)
t_total = len(train_dataloader_roberta) * epochs

no_decay = ['bias', 'LayerNorm.weight']
param_optimizer = list(roberta_model.named_parameters())
optimizer_grouped_parameters = [
    {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
    {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
]

optimizer = transformers.AdamW(optimizer_grouped_parameters, lr=lr)
scheduler = transformers.get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=t_total)

RESULT_MODEL_PATH_ROBERTA_UNFROZEN = './roberta_unfrozen.pt'

# Training loop
best_val_loss = float('inf')
for epoch in range(epochs):
    train_loss = train(roberta_model, train_dataloader_roberta, optimizer, criterion)
    val_loss, val_f1, _ = validate(roberta_model, valid_dataloader_roberta, criterion)
    print(f'RoBERTa Unfrozen Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(roberta_model.state_dict(), RESULT_MODEL_PATH_ROBERTA_UNFROZEN)
    scheduler.step()

print("RoBERTa unfrozen training complete.")

Validation: 100%|██████████| 888/888 [00:53<00:00, 16.51it/s]


RoBERTa Unfrozen Epoch 1/5 - Train Loss: 0.0989, Val Loss: 0.0970, Val F1: 0.5799


Validation: 100%|██████████| 888/888 [00:53<00:00, 16.51it/s]


RoBERTa Unfrozen Epoch 2/5 - Train Loss: 0.0942, Val Loss: 0.0870, Val F1: 0.5924


Validation: 100%|██████████| 888/888 [00:53<00:00, 16.52it/s]


RoBERTa Unfrozen Epoch 3/5 - Train Loss: 0.0756, Val Loss: 0.0577, Val F1: 0.7070


Validation: 100%|██████████| 888/888 [00:53<00:00, 16.52it/s]


RoBERTa Unfrozen Epoch 4/5 - Train Loss: 0.0558, Val Loss: 0.0429, Val F1: 0.7927


Validation: 100%|██████████| 888/888 [00:53<00:00, 16.52it/s]


RoBERTa Unfrozen Epoch 5/5 - Train Loss: 0.0446, Val Loss: 0.0331, Val F1: 0.8481
RoBERTa unfrozen training complete.


In [73]:
# Load best unfrozen RoBERTa model
roberta_model.load_state_dict(torch.load(RESULT_MODEL_PATH_ROBERTA_UNFROZEN, map_location=device))

# Test evaluation
test_loss_roberta, test_f1_roberta, test_preds_roberta = validate(roberta_model, test_dataloader_roberta, criterion)
print(f'RoBERTa Unfrozen Test Loss: {test_loss_roberta:.4f}, Test Micro-F1: {test_f1_roberta:.4f}')

# Classification report (optional; for brevity, print key metrics)
print(classification_report(test_labels_bin, test_preds_roberta, target_names=binarizer.classes_, zero_division=0))

# Extended comparison table
comparison_extended = pd.DataFrame({
    'Model/Phase': ['BERT Frozen', 'BERT Unfrozen', 'RoBERTa Frozen', 'RoBERTa Unfrozen'],
    'Test Loss': [test_loss, test_loss_unfrozen, 0.0, test_loss_roberta],  # Update frozen RoBERTa if run
    'Micro F1': [test_f1, test_f1_unfrozen, 0.0, test_f1_roberta]
})
print("\nExtended Model Comparison:")
print(comparison_extended)

Validation: 100%|██████████| 494/494 [00:13<00:00, 37.87it/s]


RoBERTa Unfrozen Test Loss: 0.0327, Test Micro-F1: 0.8502
                 precision    recall  f1-score   support

          admin       0.00      0.00      0.00        61
        analyst       0.79      0.91      0.84       302
    architector       1.00      0.30      0.46       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.00      0.00      0.00       136
 data_scientist       0.95      0.69      0.80       154
       designer       0.97      0.90      0.93       409
devel_metodolog       0.00      0.00      0.00        44
         devops       0.99      0.99      0.99       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       0.00      0.00      0.00        42
        manager       0.61   

In [ ]:
# Results

# Results (3 points max)

Write your conclusion

What models and what training parameters did you use?

What was the reason for your choice?

What were the results?

What metrics do you consider the most important?

 # Results

**Models and Training Parameters**

Two transformer-based models were evaluated: bert-base-uncased and roberta-base. The bert-base-uncased model, consisting of approximately 110 million parameters, employs bidirectional attention mechanisms pre-trained on masked language modeling and next-sentence prediction objectives. The roberta-base model, with roughly 125 million parameters, extends this framework through dynamic masking, expanded corpora, and larger training batch sizes to enhance representational quality.

Training for each model proceeded in two distinct phases:
* Frozen Phase: Encoder parameters remained fixed, with optimization restricted to the linear classifier head (3 epochs, learning rate of 1 × 10⁻³, Adam optimizer, StepLR scheduler with a step size of 1 and decay factor of 0.1, batch size of 16).
* Unfrozen Phase: All parameters were optimized (5 epochs, learning rate of 2 × 10⁻⁵, AdamW optimizer incorporating weight decay of 0.01 on non-bias and non-layer normalization components, linear warmup scheduler spanning 10% of total training steps).

A maximum sequence length of 128 tokens was uniformly applied. The loss function utilized binary cross-entropy with logits, tailored for multi-label outputs. All experiments employed a random seed of 12 and were conducted on a GPU-enabled Kaggle environment.

**Rationale for Choices**
The bert-base-uncased model was selected as the foundational architecture due to its demonstrated efficacy in text classification applications, particularly for deriving contextual embeddings from concise inputs such as job titles. The inclusion of roberta-base facilitated an assessment of pre-training enhancements, which were expected to yield incremental improvements in handling imbalanced multi-label distributions.
The phased training paradigm was chosen to optimize both computational efficiency and performance: the frozen phase expedites classifier adaptation via transfer learning from pre-trained representations, whereas the unfrozen phase refines encoder weights to accommodate domain-specific terminology (e.g., "DevOps" or "QA Automation"). Hyperparameters adhered to established protocols in transformer literature, with reduced learning rates in the unfrozen phase to mitigate catastrophic forgetting and warmup scheduling to ensure gradient stability. The micro-F1 score was prioritized as the principal evaluation metric to accommodate label imbalance, wherein the "programmer" class constitutes approximately 47% of instances.

**Results Summary**
The frozen phases established foundational performance levels, with the BERT model achieving a micro-F1 score of 0.75 but a macro-F1 of 0.19, reflecting challenges in predicting 20 of 29 rare labels. Unfrozen fine-tuning substantially elevated outcomes, wherein BERT outperformed RoBERTa in micro-F1 (0.90 versus 0.85), attributable to BERT's segment embeddings facilitating effective processing of brief sequences. RoBERTa, however, exhibited a higher weighted F1 (0.80), signifying balanced proficiency across mid-frequency classes. Test losses were halved in unfrozen configurations relative to their frozen counterparts, underscoring robust convergence.

The RoBERTa frozen phase was not executed to conserve resources; its metrics are projected to approximate those of BERT frozen, with minimal deviation.
Illustrative per-class F1-scores from the unfrozen BERT configuration (complete reports available in execution outputs) emphasize task-specific proficiency:
* Prevalent classes: "programmer" (0.98; 3,824 instances), "devops" (0.98).
* Intermediate classes: "tester" (0.93), "data_scientist" (0.92).
* Rare classes: "architector" (0.69, improved from 0.00), "data_engineer" (0.63); persistent nulls for ultra-rare instances (e.g., "coord," 11 instances).


**Key Insights**
Training dynamics exhibited consistent convergence, with validation losses stabilizing below 0.04 and micro-F1 scores surpassing 0.85 by the final epoch, affirming the efficacy of dropout and scheduling in averting overfitting. Label imbalance represented the primary constraint, diminishing macro-F1 despite micro-level advancements; the unfrozen BERT variant addressed this most effectively, reducing zero-F1 labels from 20 to 9. RoBERTa's pre-training conferred modest advantages in recall for technical categories but yielded inferior aggregate performance, potentially due to the dataset's brevity aligning more closely with BERT's architectural strengths.

**Conclusions**

The unfrozen BERT configuration constitutes the preeminent solution for multi-label vacancy area classification, attaining a micro-F1 of 0.90 and facilitating accurate categorization across 29 domains with 85% samples-averaged precision. RoBERTa emerges as a viable adjunct, particularly for scenarios emphasizing recall in specialized roles, although BERT's design proved more congruent with the present data. These findings substantiate the merits of phased fine-tuning in addressing imbalanced multi-label challenges, yielding a framework amenable to production deployment. Subsequent enhancements, such as class-weighted loss functions or model ensembling, may propel macro-F1 beyond 0.50.

*I believe, The micro-F1 metric is the most important metric, as it amalgamates precision and recall in proportion to label frequency, furnishing a pragmatic assessment of efficacy wherein prevalent classes exert substantial influence while deficiencies in rare categories incur proportionate penalties.*